# M13. 원-핫 인코딩

> 📌 **언제 필요한가**  
> 범주형 변수(글자/카테고리)를 모델에 넣을 때.  
> 예: `'서울', '부산', '인천'` → 모델은 숫자만 받아요.

## 이 모듈에서 배울 것

- 범주형 데이터의 문제점
- `pd.get_dummies` 사용
- 4차시 펭귄에서 본 그것 한 번 더 정리
- 다중공선성 회피 (`drop_first`)

---


## 1. 왜 원-핫 인코딩?

모델은 수치만 받아요. 그런데 데이터엔 카테고리(글자)가 자주 있어요:


In [ ]:
import pandas as pd

df = pd.DataFrame({
    '이름': ['철수', '영희', '민수', '지영'],
    '도시': ['서울', '부산', '서울', '인천'],
    '점수': [85, 70, 90, 75]
})
df


`도시`를 어떻게 모델에 넣을까요?

**나쁜 방법**: 그냥 숫자로 매핑 (서울=1, 부산=2, 인천=3)
- 문제: "인천=3은 서울=1의 3배" 같은 의미가 생김. 실제론 그런 관계 없음.

**좋은 방법**: **원-핫 인코딩** — 각 카테고리를 0/1 컬럼으로


## 2. `pd.get_dummies` — 한 줄로 끝


In [ ]:
df_encoded = pd.get_dummies(df, columns=['도시'])
df_encoded


**결과 해석**:
- 철수 → `도시_서울=1`, `도시_부산=0`, `도시_인천=0`
- 영희 → `도시_부산=1`, 나머지 0
- 등...

각 행은 카테고리 중 정확히 하나만 1.


## 3. 진짜 활용 예시 — 펭귄 종

4차시 보너스에서 봤던 그것:


In [ ]:
import seaborn as sns

# Colab에서는 직접 받기 가능
# penguins = sns.load_dataset('penguins').dropna()

# 여기선 합성 데이터로 시연
penguins = pd.DataFrame({
    'species': ['Adelie', 'Adelie', 'Chinstrap', 'Gentoo', 'Gentoo'],
    'bill_length_mm': [39.1, 39.5, 46.5, 46.1, 50.0],
    'body_mass_g': [3750, 3800, 3500, 4500, 5700]
})
penguins


In [ ]:
# species 원-핫 인코딩
penguins_encoded = pd.get_dummies(penguins, columns=['species'])
penguins_encoded


이제 `species_Adelie`, `species_Chinstrap`, `species_Gentoo` 세 컬럼이 0/1로 모델 입력 가능.


## 4. `drop_first` — 다중공선성 회피


In [ ]:
# drop_first=True: 첫 카테고리 제거 (k-1개 컬럼)
penguins_encoded_v2 = pd.get_dummies(penguins, columns=['species'], drop_first=True)
penguins_encoded_v2


**왜 첫 번째 카테고리를 빼나?**

세 컬럼 다 두면 정보 중복:
- `Adelie=0, Chinstrap=0` 이면 자동으로 Gentoo
- 즉 한 컬럼은 다른 두 개로부터 결정됨 (**다중공선성**)

회귀 모델에선 이걸 빼야 안정적이에요. 단, 신경망에선 보통 다 둬도 OK.


## 5. 여러 카테고리 컬럼 한 번에


In [ ]:
example = pd.DataFrame({
    '도시': ['서울', '부산'],
    '성별': ['M', 'F'],
    '학년': ['1학년', '2학년'],
    '점수': [85, 70]
})

# 컬럼 여러 개 지정
encoded = pd.get_dummies(example, columns=['도시', '성별', '학년'])
encoded


## 6. 본인 데이터에 적용해보기 ✏️


In [ ]:
# 카테고리 컬럼 자동 탐지
# cat_cols = my_df.select_dtypes(include='object').columns.tolist()
# 
# # 원-핫
# my_df_encoded = pd.get_dummies(my_df, columns=cat_cols)


## 7. ⚠️ 함정 / 주의사항

### 7.1 카테고리 수가 너무 많으면 컬럼 폭증
'시군구'처럼 카테고리 250개면 → 컬럼 250개 추가.  
**해결**: 그룹화 (대분류로 묶기), 또는 빈도 낮은 건 'Other'로.

### 7.2 훈련 데이터에 없는 카테고리 등장
훈련 시 안 본 새 카테고리는 처리 불가.  
**해결**: `sklearn.preprocessing.OneHotEncoder`에 `handle_unknown='ignore'`.

### 7.3 데이터 타입
`get_dummies`의 결과는 boolean. 일부 모델에서 정수가 필요하면:
```python
pd.get_dummies(df).astype(int)
```

### 7.4 카테고리 순서가 의미 있는 경우 (Ordinal)
`'하/중/상'`처럼 순서가 있으면 원-핫 대신 정수 매핑:
```python
df['등급_num'] = df['등급'].map({'하': 1, '중': 2, '상': 3})
```


## 8. 📚 더 알아보기

- `sklearn.preprocessing.OneHotEncoder` — 훈련/테스트 일관 처리
- `pd.Categorical(values, ordered=True)` — 순서 있는 카테고리
- 빈도 낮은 카테고리 그룹화: `value_counts()` 후 임계치 이하는 'Other'로
